# 🛡️ VAHAAN Road Safety — Multi-Model Training Pipeline

Trains 4 safety detection models from your HuggingFace datasets:
| Model | Target | Output |
|---|---|---|
| `helmet_net.pt` | Helmet / No-Helmet on motorcycles | `Road safety data/helmet_motorcycle/` |
| `seatbelt_net.pt` | Seat belt detection in cars | `Road safety data/seatbelt/` |
| `phone_net.pt` | Phone use while driving | `Road safety data/phone safety/` |
| `pothole_net.pt` | Pothole & road crack detection | `Road safety data/india.v2i.yolov11/` |

**Instructions:**
1. Upload your datasets to HuggingFace as a dataset repo
2. Set `HF_REPO` below to your repo name
3. Enable GPU (P100 recommended) in Kaggle → Settings → Accelerator
4. Run All

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
HF_REPO        = "YOUR_HF_USERNAME/vahaan-road-safety-datasets"  # ← change this
HF_TOKEN       = ""    # Only needed if repo is private — add as Kaggle Secret

BASE_MODEL     = "yolov8n.pt"   # yolov8n=fastest, yolov8s=balanced, yolov8m=accurate
EPOCHS         = 80
IMGSZ          = 640
BATCH          = 16             # Reduce to 8 if OOM
PATIENCE       = 20             # Early stop if no improvement

# Dataset folder names inside your HF repo
DATASETS = {
    "helmet":   "helmet_motorcycle",
    "seatbelt": "seatbelt",
    "phone":    "phone safety",
    "pothole":  "india.v2i.yolov11",
}
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── INSTALL DEPENDENCIES ─────────────────────────────────────────────────────
!pip install -q ultralytics huggingface_hub
print('✅ Dependencies installed')

In [ ]:
# ── DOWNLOAD DATASETS FROM HUGGINGFACE ───────────────────────────────────────
import os
from huggingface_hub import snapshot_download

DATA_ROOT = "/kaggle/working/road_safety_data"
os.makedirs(DATA_ROOT, exist_ok=True)

print(f'📥 Downloading datasets from HuggingFace: {HF_REPO}')
snapshot_download(
    repo_id   = HF_REPO,
    repo_type = "dataset",
    local_dir = DATA_ROOT,
    token     = HF_TOKEN or None,
)
print('✅ All datasets downloaded')

# Verify all expected folders are present
for name, folder in DATASETS.items():
    path = os.path.join(DATA_ROOT, folder)
    yaml = os.path.join(path, 'data.yaml')
    exists = '✅' if os.path.exists(yaml) else '❌ MISSING'
    imgs   = len([f for f in os.listdir(os.path.join(path,'train','images')) if f.endswith(('.jpg','.png'))]) if os.path.exists(os.path.join(path,'train','images')) else 0
    print(f'  {exists} [{name}] {folder} — {imgs} train images')

In [ ]:
# ── FIX data.yaml PATHS (Kaggle paths differ from local) ─────────────────────
import yaml

def fix_yaml_paths(yaml_path, data_root):
    """Update train/val/test paths in data.yaml to absolute Kaggle paths."""
    with open(yaml_path, 'r') as f:
        cfg = yaml.safe_load(f)

    dataset_dir = os.path.dirname(yaml_path)
    for split in ['train', 'val', 'test']:
        if split in cfg:
            cfg[split] = os.path.join(dataset_dir, split, 'images')

    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f)

    print(f'  ✅ Patched: {yaml_path}')
    return yaml_path

yaml_paths = {}
print('🔧 Patching data.yaml paths for Kaggle environment...')
for name, folder in DATASETS.items():
    yp = os.path.join(DATA_ROOT, folder, 'data.yaml')
    if os.path.exists(yp):
        yaml_paths[name] = fix_yaml_paths(yp, DATA_ROOT)
    else:
        print(f'  ❌ Missing: {yp}')

In [ ]:
# ── TRAINING FUNCTION ─────────────────────────────────────────────────────────
from ultralytics import YOLO
import shutil

OUTPUT_DIR = '/kaggle/working/vahaan_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAINING_RESULTS = {}

def train_model(name, yaml_path, output_name):
    print(f'\n{"="*60}')
    print(f'  🚀 Training: {output_name}')
    print(f'  Data: {yaml_path}')
    print(f'{"="*60}')

    model   = YOLO(BASE_MODEL)
    results = model.train(
        data     = yaml_path,
        epochs   = EPOCHS,
        imgsz    = IMGSZ,
        batch    = BATCH,
        name     = output_name,
        patience = PATIENCE,
        device   = 0,
        augment  = True,
        # Indian road augmentation profile
        hsv_s    = 0.7,
        hsv_v    = 0.4,
        flipud   = 0.05,
        mosaic   = 1.0,
    )

    # Copy best weights to output
    best = f'runs/detect/{output_name}/weights/best.pt'
    dest = os.path.join(OUTPUT_DIR, f'{output_name}.pt')
    if os.path.exists(best):
        shutil.copy(best, dest)
        print(f'✅ Saved: {dest}')
    else:
        print(f'⚠️ best.pt not found for {output_name}')

    TRAINING_RESULTS[name] = {
        'output': dest,
        'map50':  results.results_dict.get('metrics/mAP50(B)', 'N/A'),
        'map5095': results.results_dict.get('metrics/mAP50-95(B)', 'N/A'),
    }
    return results

In [ ]:
# ── TRAIN ALL 4 MODELS ────────────────────────────────────────────────────────
# Each model is independent — if one fails the rest still run

MODEL_CONFIGS = [
    ('helmet',   'helmet_net'),
    ('seatbelt', 'seatbelt_net'),
    ('phone',    'phone_net'),
    ('pothole',  'pothole_net'),
]

for ds_name, model_name in MODEL_CONFIGS:
    if ds_name in yaml_paths:
        try:
            train_model(ds_name, yaml_paths[ds_name], model_name)
        except Exception as e:
            print(f'❌ Failed to train {model_name}: {e}')
    else:
        print(f'⏭️ Skipping {model_name} — dataset not found')

In [ ]:
# ── FINAL RESULTS SUMMARY ────────────────────────────────────────────────────
print('\n' + '='*60)
print('  🏁 VAHAAN TRAINING COMPLETE')
print('='*60)
for name, result in TRAINING_RESULTS.items():
    map50   = result['map50']
    map5095 = result['map5095']
    print(f"  [{name:10}]  mAP50: {map50:.3f}  |  mAP50-95: {map5095:.3f}  →  {result['output']}")

print('\n📁 All models saved to:', OUTPUT_DIR)
print('📥 Download from: Kaggle → Output → vahaan_models/')
print('\nNext: Drop .pt files into your VAHAAN project root and restart the API.')

In [ ]:
# ── OPTIONAL: PUSH TRAINED MODELS BACK TO HUGGINGFACE ──────────────────────
# Uncomment to auto-upload trained weights to HuggingFace as a model repo

# from huggingface_hub import HfApi
# api = HfApi()
# MODEL_REPO = f"{HF_REPO.split('/')[0]}/vahaan-safety-models"
# api.create_repo(MODEL_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
# for f in os.listdir(OUTPUT_DIR):
#     if f.endswith('.pt'):
#         api.upload_file(
#             path_or_fileobj=os.path.join(OUTPUT_DIR, f),
#             path_in_repo=f,
#             repo_id=MODEL_REPO,
#             repo_type='model',
#             token=HF_TOKEN
#         )
#         print(f'✅ Uploaded: {f} → {MODEL_REPO}')